In [1]:
!python --version && java -version && pip show pyspark | head -2 && free -h && df -h /content | tail -1

Python 3.13.18
[0.019s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.019s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
Name: pyspark
Version: 4.0.5
ERROR: Pipe to stdout was broken
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.0Gi       9.2Gi       2.2Mi       2.8Gi        11Gi
Swap:             0B          0B          0B
overlay         108G   85M  108G   1% /


In [2]:
!curl -s https://dlcdn.apache.org/hadoop/common/stable/ | grep -o 'hadoop-[0-9.]*\.tar\.gz' | sort -u

hadoop-3.5.0.tar.gz


In [3]:
import os, subprocess
!wget -q https://dlcdn.apache.org/hadoop/common/stable/hadoop-3.5.0.tar.gz
!tar -xzf hadoop-3.5.0.tar.gz -C /content
java_bin = subprocess.check_output("readlink -f $(which java)", shell=True, text=True).strip()
os.environ["JAVA_HOME"] = java_bin.rsplit("/bin/java", 1)[0]
os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
os.environ["PATH"] += ":/content/hadoop-3.5.0/bin"
print("JAVA_HOME =", os.environ["JAVA_HOME"])
!hadoop version

JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64
[0.028s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.028s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.5.0
Source code repository https://github.com/apache/hadoop.git -r dbcc7cd797100e6b32cd84f85b53a5193a5f9af0
Compiled by cnauroth on 2026-03-24T16:56Z
Compiled on platform linux-x86_64
Compiled with protoc 3.25.5
From source with checksum 64582ed0a62a169c39a19c734a8f33b5
This command was run using /content/hadoop-3.5.0/share/hadoop/common/hadoop-common-3.5.0.jar


In [4]:
!git clone https://github.com/usamahanif349/pk-footwear-budget-optimizer.git
%cd /content/pk-footwear-budget-optimizer
!python src/simulation/generate_data.py --rows 10000
!python src/simulation/generate_data.py --rows 100000
!python src/simulation/generate_data.py --rows 1000000
!python src/simulation/generate_data.py --rows 10000000
!ls -lh data/raw

Cloning into 'pk-footwear-budget-optimizer'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 53 (delta 5), reused 47 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 187.39 KiB | 4.93 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/pk-footwear-budget-optimizer
Saved 10,000 rows -> /content/pk-footwear-budget-optimizer/data/raw/ads_10000.csv
              spend   revenue  ROAS
platform                           
Facebook   16099953  47070938  2.92
Instagram  13359120  33408421  2.50
Google     10986691  23987716  2.18
TikTok      7930093  20054829  2.53
YouTube     6245577   7549808  1.21
Saved 100,000 rows -> /content/pk-footwear-budget-optimizer/data/raw/ads_100000.csv
               spend    revenue  ROAS
platform                             
Facebook   154618613  456754799  2.95
Instagram  138253804  337540092  2.44
Google     110961361  233846921  2.11
T

In [5]:
import os
os.makedirs("src/mapreduce", exist_ok=True)

mapper = r'''#!/usr/bin/env python3
# MAP: read CSV lines, emit key=platform|campaign, value=5 sums.
# Uses the "in-mapper combining" pattern: sums inside each mapper first,
# so far less data goes through the shuffle.
import sys
acc = {}
for line in sys.stdin:
    f = line.rstrip("\n").split(",")
    if f[0] == "date":          # skip header
        continue
    key = f[2] + "|" + f[1]     # platform|campaign
    v = acc.get(key)
    if v is None:
        v = acc[key] = [0, 0, 0, 0, 0]
    v[0] += int(f[3]); v[1] += int(f[4]); v[2] += int(f[5])
    v[3] += int(f[6]); v[4] += int(f[7])
for key, v in acc.items():
    sys.stdout.write(key + "\t" + ",".join(map(str, v)) + "\n")
'''

reducer = r'''#!/usr/bin/env python3
# REDUCE: input arrives sorted by key; add up the 5 sums per key.
import sys
cur, acc = None, [0, 0, 0, 0, 0]
for line in sys.stdin:
    key, vals = line.rstrip("\n").split("\t")
    v = [int(x) for x in vals.split(",")]
    if key != cur:
        if cur is not None:
            sys.stdout.write(cur + "\t" + ",".join(map(str, acc)) + "\n")
        cur, acc = key, [0, 0, 0, 0, 0]
    acc = [a + b for a, b in zip(acc, v)]
if cur is not None:
    sys.stdout.write(cur + "\t" + ",".join(map(str, acc)) + "\n")
'''

open("src/mapreduce/mapper.py", "w").write(mapper)
open("src/mapreduce/reducer.py", "w").write(reducer)

print("--- Test 1: plain Python pipe (no Hadoop) ---")
!cat data/raw/ads_10000.csv | python3 src/mapreduce/mapper.py | sort | python3 src/mapreduce/reducer.py | head -5

print("--- Test 2: real Hadoop MapReduce ---")
!rm -rf /content/mr_out
!hadoop jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-3.5.0.jar -D mapreduce.local.map.tasks.maximum=2 -files src/mapreduce/mapper.py,src/mapreduce/reducer.py -mapper "python3 mapper.py" -reducer "python3 reducer.py" -input data/raw/ads_10000.csv -output /content/mr_out 2>&1 | grep -E "completed successfully|ERROR|Exception" | head -5
!head -5 /content/mr_out/part-*

--- Test 1: plain Python pipe (no Hadoop) ---
Facebook|11.11 Mega Sale	15068531,118316,2277751,2417,10833508
Facebook|Always-On Brand	36892074,206206,5531317,1904,8655420
Facebook|Back to School	11000506,72347,1638937,1080,4911947
Facebook|Eid ul Adha Sale	10085066,72823,1520194,1311,5977832
Facebook|Eid ul Fitr Sale	12540713,94092,1837905,1820,8217665
--- Test 2: real Hadoop MapReduce ---
		IO_ERROR=0
2026-11-04 21:13:20,955 INFO mapreduce.Job: Job job_local1277416303_0001 completed successfully
		IO_ERROR=0
Facebook|11.11 Mega Sale	15068531,118316,2277751,2417,10833508
Facebook|Always-On Brand	36892074,206206,5531317,1904,8655420
Facebook|Back to School	11000506,72347,1638937,1080,4911947
Facebook|Eid ul Adha Sale	10085066,72823,1520194,1311,5977832
Facebook|Eid ul Fitr Sale	12540713,94092,1837905,1820,8217665


In [6]:
import os, time, glob, shutil, subprocess, statistics
import numpy as np
import pandas as pd
import plotly.express as px
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType

SIZES = [10_000, 100_000, 1_000_000, 10_000_000]
REPEATS = 3
KEYS = ["platform", "campaign"]
COLS = ["impressions", "clicks", "spend", "conversions", "revenue"]

# ---------- Pandas ----------
def run_pandas(path):
    df = pd.read_csv(path, usecols=KEYS + COLS)
    return df.groupby(KEYS)[COLS].sum().sort_index()

# ---------- Spark ----------
spark = (SparkSession.builder.master("local[*]").appName("bench")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.shuffle.partitions", "8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
SCHEMA = StructType(
    [StructField("date", StringType()), StructField("campaign", StringType()),
     StructField("platform", StringType())]
    + [StructField(c, LongType()) for c in COLS]
    + [StructField(c, StringType()) for c in ["age", "gender", "city", "device"]])

def run_spark(path):
    rows = (spark.read.csv(path, header=True, schema=SCHEMA)
            .groupBy(KEYS).agg(*[F.sum(c).alias(c) for c in COLS]).collect())
    return pd.DataFrame([r.asDict() for r in rows]).set_index(KEYS).sort_index()

# ---------- Hadoop MapReduce (streaming, local mode) ----------
JAR = f"{os.environ['HADOOP_HOME']}/share/hadoop/tools/lib/hadoop-streaming-3.5.0.jar"

def run_hadoop(path):
    shutil.rmtree("/content/mr_out", ignore_errors=True)
    cmd = ["hadoop", "jar", JAR, "-D", "mapreduce.local.map.tasks.maximum=2",
           "-files", "src/mapreduce/mapper.py,src/mapreduce/reducer.py",
           "-mapper", "python3 mapper.py", "-reducer", "python3 reducer.py",
           "-input", path, "-output", "/content/mr_out"]
    subprocess.run(cmd, check=True, capture_output=True)
    rows = []
    for fn in glob.glob("/content/mr_out/part-*"):
        for line in open(fn):
            key, vals = line.rstrip("\n").split("\t")
            plat, camp = key.split("|")
            rows.append([plat, camp] + [int(x) for x in vals.split(",")])
    return pd.DataFrame(rows, columns=KEYS + COLS).set_index(KEYS).sort_index()

ENGINES = [("Pandas", run_pandas), ("Hadoop MapReduce", run_hadoop), ("Spark", run_spark)]

# warm-up (JVM start-up is not counted)
run_spark("data/raw/ads_10000.csv")

results = []
for n in SIZES:
    path = f"data/raw/ads_{n}.csv"
    outs = {}
    for name, fn in ENGINES:
        times = []
        for _ in range(REPEATS):
            t0 = time.perf_counter()
            out = fn(path)
            times.append(time.perf_counter() - t0)
        outs[name] = out
        med = statistics.median(times)
        results.append(dict(rows=n, engine=name, median_s=med, min_s=min(times),
                            max_s=max(times), runs=REPEATS, rows_per_s=n / med))
        print(f"{n:>11,} rows | {name:17s} | median {med:7.2f}s")
    ref = outs["Pandas"]
    same = all(o.index.equals(ref.index) and np.array_equal(o[COLS].to_numpy(), ref[COLS].to_numpy())
               for o in outs.values())
    print(f"            -> all 3 engines give identical totals: {same}\n")
    for r in results[-3:]:
        r["identical_output"] = same

bench = pd.DataFrame(results)
os.makedirs("data/processed", exist_ok=True)
bench.to_csv("data/processed/benchmark_results.csv", index=False)
print(bench.pivot(index="rows", columns="engine", values="median_s").round(2))

fig = px.line(bench, x="rows", y="median_s", color="engine", markers=True,
              log_x=True, log_y=True, title="Read + aggregate time (log-log)")
fig.show()

     10,000 rows | Pandas            | median    0.01s
     10,000 rows | Hadoop MapReduce  | median    4.58s
     10,000 rows | Spark             | median    1.07s
            -> all 3 engines give identical totals: True

    100,000 rows | Pandas            | median    0.09s
    100,000 rows | Hadoop MapReduce  | median    4.62s
    100,000 rows | Spark             | median    0.93s
            -> all 3 engines give identical totals: True

  1,000,000 rows | Pandas            | median    0.71s
  1,000,000 rows | Hadoop MapReduce  | median    7.54s
  1,000,000 rows | Spark             | median    2.09s
            -> all 3 engines give identical totals: True

 10,000,000 rows | Pandas            | median    7.55s
 10,000,000 rows | Hadoop MapReduce  | median   24.83s
 10,000,000 rows | Spark             | median   13.54s
            -> all 3 engines give identical totals: True

engine    Hadoop MapReduce  Pandas  Spark
rows                                     
10000                 4.

In [7]:
import gc, resource, psutil

# 1) build the 100M-row file (header + 10 copies of the 10M body)
BIG = "data/raw/ads_100000000.csv"
!head -1 data/raw/ads_10000000.csv > {BIG}
!for i in $(seq 10); do tail -n +2 data/raw/ads_10000000.csv >> {BIG}; done
!ls -lh {BIG}

# expected totals = 10 x the 10M totals
exp_spend = int(outs["Pandas"]["spend"].sum()) * 10
exp_rev = int(outs["Pandas"]["revenue"].sum()) * 10
del outs, out
gc.collect()

# 2) pandas job runs in its own process
open("/content/pandas_job.py", "w").write(r'''
import sys, time, resource
import pandas as pd
path, mode = sys.argv[1], sys.argv[2]
KEYS = ["platform", "campaign"]
COLS = ["impressions", "clicks", "spend", "conversions", "revenue"]
t0 = time.perf_counter()
if mode == "plain":
    out = pd.read_csv(path, usecols=KEYS + COLS).groupby(KEYS)[COLS].sum()
else:
    parts = [c.groupby(KEYS)[COLS].sum()
             for c in pd.read_csv(path, usecols=KEYS + COLS, chunksize=2_000_000)]
    out = pd.concat(parts).groupby(level=[0, 1]).sum()
t = time.perf_counter() - t0
peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**2
print(f"OK {t:.2f} {peak:.2f} {int(out['spend'].sum())} {int(out['revenue'].sum())}")
''')

CAP = int(psutil.virtual_memory().available - 1.5 * 1024**3)
print(f"Memory cap for pandas: {CAP / 1024**3:.1f} GB")

def run_pandas_capped(mode):
    def limit():
        resource.setrlimit(resource.RLIMIT_AS, (CAP, CAP))
    p = subprocess.run(["python3", "/content/pandas_job.py", BIG, mode],
                       preexec_fn=limit, capture_output=True, text=True)
    if p.returncode == 0 and p.stdout.strip().startswith("OK"):
        _, t, peak, s, r = p.stdout.split()
        return dict(status="OK", seconds=float(t), peak_gb=float(peak),
                    correct=(int(s) == exp_spend and int(r) == exp_rev))
    err = (p.stderr.strip().splitlines() or ["killed"])[-1][:120]
    return dict(status="FAILED", seconds=None, peak_gb=None, correct=None, error=err)

def timed(fn):
    t0 = time.perf_counter()
    o = fn(BIG)
    t = time.perf_counter() - t0
    ok = int(o["spend"].sum()) == exp_spend and int(o["revenue"].sum()) == exp_rev
    return dict(status="OK", seconds=t, peak_gb=None, correct=ok)

big_results = []
for name, job in [("Pandas", lambda: run_pandas_capped("plain")),
                  ("Pandas (chunked)", lambda: run_pandas_capped("chunked")),
                  ("Spark", lambda: timed(run_spark)),
                  ("Hadoop MapReduce", lambda: timed(run_hadoop))]:
    print(f"running {name} ...", flush=True)
    r = job()
    r["engine"] = name
    big_results.append(r)
    print("  ", r, flush=True)

big = pd.DataFrame(big_results)
big["rows"] = 100_000_000
big["memory_cap_gb"] = round(CAP / 1024**3, 1)
big["data_note"] = "replicated 10x from the 10M file"
big.to_csv("data/processed/benchmark_100M.csv", index=False)
print(big[["engine", "status", "seconds", "peak_gb", "correct"]])

-rw-r--r-- 1 root root 8.0G Nov  4 21:23 data/raw/ads_100000000.csv
Memory cap for pandas: 7.8 GB
running Pandas ...
   {'status': 'FAILED', 'seconds': None, 'peak_gb': None, 'correct': None, 'error': 'numpy._core._exceptions._ArrayMemoryError: Unable to allocate 763. MiB for an array with shape (100000000,) and data typ', 'engine': 'Pandas'}
running Pandas (chunked) ...
   {'status': 'OK', 'seconds': 82.79, 'peak_gb': 0.49, 'correct': True, 'engine': 'Pandas (chunked)'}
running Spark ...
   {'status': 'OK', 'seconds': 137.26199387099996, 'peak_gb': None, 'correct': True, 'engine': 'Spark'}
running Hadoop MapReduce ...
   {'status': 'OK', 'seconds': 212.73367263499995, 'peak_gb': None, 'correct': True, 'engine': 'Hadoop MapReduce'}
             engine  status     seconds  peak_gb correct
0            Pandas  FAILED         NaN      NaN    None
1  Pandas (chunked)      OK   82.790000     0.49    True
2             Spark      OK  137.261994      NaN    True
3  Hadoop MapReduce      OK  2

In [8]:
from google.colab import files
for f in ["data/processed/benchmark_results.csv",
          "data/processed/benchmark_100M.csv",
          "src/mapreduce/mapper.py",
          "src/mapreduce/reducer.py"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>